In [3]:
import h5py

H5 = "/scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha0_spectrum.h5"

def dump_h5_tree(h5_path: str, max_attrs: int = 30):
    def _show_attrs(obj, indent=""):
        try:
            keys = list(obj.attrs.keys())[:max_attrs]
            if keys:
                print(f"{indent}  attrs({len(obj.attrs)}): {keys}")
        except Exception:
            pass

    with h5py.File(h5_path, "r") as f:
        print("FILE:", h5_path)
        print("TOP-LEVEL KEYS:", list(f.keys()))
        _show_attrs(f)

        def visitor(name, obj):
            kind = "GROUP" if isinstance(obj, h5py.Group) else "DATASET"
            if isinstance(obj, h5py.Dataset):
                print(f"{kind:7s} /{name}  shape={obj.shape}  dtype={obj.dtype}")
            else:
                print(f"{kind:7s} /{name}")
            _show_attrs(obj, indent="")

        f.visititems(visitor)

dump_h5_tree(H5)

FILE: /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha0_spectrum.h5
TOP-LEVEL KEYS: ['meta', 'product_meta', 'spectrum']
GROUP   /meta
  attrs(15): ['RUN_LABEL', 'Rvir_kpc', 'SNAP', 'SubhaloID', 'alpha_deg', 'end_ckpch', 'inc_deg', 'instrument', 'lines', 'mode', 'phi_deg', 'rho_kpc', 'sightline_id', 'start_ckpch', 'total_len_Rvir']
GROUP   /product_meta
  attrs(5): ['add_noise', 'apply_lsf', 'instrument', 'noise_seed', 'snr']
GROUP   /spectrum
GROUP   /spectrum/lsf
DATASET /spectrum/lsf/flux  shape=(30001,)  dtype=float64
DATASET /spectrum/lsf/lambda_A  shape=(30001,)  dtype=float64
DATASET /spectrum/lsf/tau  shape=(30001,)  dtype=float64
GROUP   /spectrum/raw
DATASET /spectrum/raw/flux  shape=(30001,)  dtype=float64
DATASET /spectrum/raw/lambda_A  shape=(30001,)  dtype=float64
DATASET /spectrum/raw/tau  shape=(30001,)  dtype=float64


In [ ]:
import os
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt

def load_spectrum_any(h5_path: str, prefer: str = "lsf"):
    """
    Your current files store spectra under:
      /spectrum/{raw,lsf}/{lambda_A, flux, tau}
    Falls back to older layouts if present.
    Returns (lam_A, flux, which).
    """
    with h5py.File(h5_path, "r") as f:
        if "spectrum" in f:
            sg = f["spectrum"]

            # preferred (lsf or raw)
            if prefer in sg and all(k in sg[prefer] for k in ("lambda_A", "flux")):
                g = sg[prefer]
                return np.asarray(g["lambda_A"]), np.asarray(g["flux"]), f"/spectrum/{prefer}"

            # fallback to the other one
            for alt in ("raw", "lsf"):
                if alt in sg and all(k in sg[alt] for k in ("lambda_A", "flux")):
                    g = sg[alt]
                    return np.asarray(g["lambda_A"]), np.asarray(g["flux"]), f"/spectrum/{alt}"

            # minimal legacy layout
            if all(k in sg for k in ("lambda_A", "flux")):
                return np.asarray(sg["lambda_A"]), np.asarray(sg["flux"]), "/spectrum"

        # legacy bundle layout
        if "bundle" in f and "spectrum" in f["bundle"]:
            sg = f["bundle"]["spectrum"]
            if all(k in sg for k in ("lambda_A", "flux")):
                return np.asarray(sg["lambda_A"]), np.asarray(sg["flux"]), "/bundle/spectrum"

    raise KeyError("No recognized spectrum datasets found in this HDF5.")

def save_ion_zoom_flux_plots(
    spectra_dir: str,
    outdir: str,
    ions,                 # list of tuples: (tag, label, lam0)
    zoom_half_A: float = 6.0,
    prefer: str = "lsf",  # "lsf" or "raw"
):
    os.makedirs(outdir, exist_ok=True)

    files = sorted(glob.glob(os.path.join(spectra_dir, "*_spectrum.h5")))
    if not files:
        raise FileNotFoundError(f"No *_spectrum.h5 files found in {spectra_dir}")

    for h5 in files:
        base = os.path.splitext(os.path.basename(h5))[0]
        try:
            lam, flux, which = load_spectrum_any(h5, prefer=prefer)
        except Exception as e:
            print(f"[FAIL] {h5}: {type(e).__name__}: {e}")
            continue

        for ion_tag, label, lam0 in ions:
            m = (lam >= lam0 - zoom_half_A) & (lam <= lam0 + zoom_half_A)
            if not np.any(m):
                continue

            fig = plt.figure()
            plt.plot(lam[m], flux[m])
            plt.xlabel("Wavelength [Å]")
            plt.ylabel("Flux")
            plt.title(f"{base} | {label} | {which}")
            out_flux = os.path.join(outdir, f"{base}__{ion_tag}__flux.png")
            plt.savefig(out_flux, dpi=200, bbox_inches="tight")
            plt.close(fig)

        print(f"[OK] {h5} ({which})")

# ---- your paths ----
SPECTRA_DIR = "/scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5"
OUTDIR      = "/scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/plot_spectra"

# ---- your ions/lines ----
IONS = [
    ("HI_1216",    "H I 1215.67", 1215.67),
    ("CII_1335",   "C II 1334.53", 1334.532),
    ("SiIII_1206", "Si III 1206.50", 1206.50),
    ("SiII_1190",  "Si II 1190.42", 1190.416),
    ("SiII_1193",  "Si II 1193.29", 1193.290),
    ("SiII_1260",  "Si II 1260.42", 1260.422),
    ("NV_1239",    "N V 1238.82", 1238.821),
    ("OI_1302",    "O I 1302.17", 1302.168),
]

save_ion_zoom_flux_plots(
    spectra_dir=SPECTRA_DIR,
    outdir=OUTDIR,
    ions=IONS,
    zoom_half_A=6.0,
    prefer="lsf",   # or "raw"
)

[OK] /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha0_spectrum.h5 (/spectrum/lsf)
[OK] /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha100_spectrum.h5 (/spectrum/lsf)
[OK] /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha101_spectrum.h5 (/spectrum/lsf)
[OK] /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha102_spectrum.h5 (/spectrum/lsf)
[OK] /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+043026_flip_alpha103_spectrum.h5 (/spectrum/lsf)
[OK] /scratch/tsingh65/m61-tng/outputs/sid488530/rays_and_spectra_sid488530_snap99_L4Rvir/spectra_h5/L4Rvir_sid488530_J122138+0